## Notebook to learn to play with tif images

In [ ]:
import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))

In [ ]:
# GET SETTINGS
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)
# display(settings)

# SET RANDOM SEEDS
np.random.seed(settings["rng_seed"])
random.seed(settings["rng_seed"])
tf.random.set_seed(settings["rng_seed"])

In [ ]:
# LOAD THE DATA
imp.reload(build_data)

(tagyear_train, 
 taglat_train, 
 taglon_train,
 tagyear_val, 
 taglat_val, 
 taglon_val, 
 ) = build_data.make_sample_list(settings)

tfds_val = build_data.build_tf_dataset(settings, tagyear_val, taglat_val, taglon_val, settings["batch_size"])
tfds_val = tfds_val.prefetch(tf.data.AUTOTUNE)

In [ ]:
imp.reload(build_model)
imp.reload(train_model)

SAVE_MODEL_DIRECTORY = "saved_models/"

model_name = SAVE_MODEL_DIRECTORY + 'model_' + settings["exp_name"] + '.h5'
model = tf.keras.models.load_model(model_name)

labels_val = [labels for _, labels in tfds_val.unbatch()]
predict_val = model.predict(tfds_val)
gc.collect()

In [ ]:
plt.plot(labels_val, predict_val, '.')
plt.plot((0,1), (0,1), '-', linewidth=1, color="gray", alpha=.5)
plt.show()